# 4.9 Spectral clustering

Spectral clustering changes the problem from raw coordinates to graph geometry.

Affinities become a Laplacian; eigenvectors of that Laplacian become coordinates where ordinary clustering can separate nonlinear shapes. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.datasets import load_digits
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import silhouette_score
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler

RNG = np.random.default_rng(7)


def cluster_ladder():
    """D1..D5 clustering ladder of rising difficulty."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.3, 0.2], [3.0, 3.0], [3.2, 2.8], [0.1, 3.1], [0.2, 2.9]])
    y1 = np.array([0, 0, 1, 1, 2, 2])
    rungs.append(("D1 hand 3 clusters", x1, y1, 3))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.7, random_state=1)
    rungs.append(("D2 clean blobs", x2, y2, 3))

    x3, y3 = make_blobs(n_samples=240, centers=3, cluster_std=1.6, random_state=2)
    transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
    x3 = x3 @ transform
    rungs.append(("D3 anisotropic + overlap", x3, y3, 3))

    iris = load_iris()
    rungs.append(("D4 Iris (real, 4-D)", iris.data, iris.target, 3))

    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    rungs.append(("D5 digits 0-3 (real, 64-D)", digits.data[keep] / 16.0, digits.target[keep], 4))

    return rungs


def scaled(X):
    return StandardScaler().fit_transform(X)


def plot_xy(X):
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=0).fit_transform(X)


def safe_silhouette(X, labels):
    if len(np.unique(labels)) < 2:
        return float("nan")
    if len(np.unique(labels)) >= len(labels):
        return float("nan")
    return float(silhouette_score(X, labels))

def rbf_affinity(X, gamma=None):
    distances = pairwise_distances(X)
    if gamma is None:
        positive = distances[distances > 0]
        sigma = np.median(positive)
        gamma = 1.0 / (2.0 * sigma * sigma)
    A = np.exp(-gamma * distances ** 2)
    np.fill_diagonal(A, 0.0)
    return A


def spectral_cluster(X, n_clusters, gamma=None, n_neighbors=None, seed=0):
    if n_neighbors is None:
        A = rbf_affinity(X, gamma=gamma)
    else:
        graph = kneighbors_graph(X, n_neighbors=n_neighbors, mode="connectivity", include_self=False)
        A = graph.maximum(graph.T).toarray()
    D = np.diag(A.sum(axis=1))
    L = D - A
    vals, vecs = np.linalg.eigh(L)
    embedding = vecs[:, :n_clusters]
    norms = np.linalg.norm(embedding, axis=1, keepdims=True)
    embedding = embedding / np.maximum(norms, 1e-12)
    labels = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed).fit_predict(embedding)
    return labels, A, L, vals, embedding


## The concept, built once on D1

For a graph affinity matrix, spectral clustering starts with

$$L=D-A,\qquad Lv=\lambda v$$

The lesson's toy graph has degree vector $[1,1,1,1]$ and Laplacian eigenvalues $[0.0,0.0,2.0,2.0]$, meaning two disconnected components.

In [ ]:

A_toy = np.array([
    [0.0, 1.0, 0.0, 0.0],
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
    [0.0, 0.0, 1.0, 0.0],
])
degrees = A_toy.sum(axis=1)
D_toy = np.diag(degrees)
L_toy = D_toy - A_toy
eigenvalues = np.linalg.eigvalsh(L_toy)

assert np.allclose(degrees, [1.0, 1.0, 1.0, 1.0])
assert np.allclose(eigenvalues, [0.0, 0.0, 2.0, 2.0])
assert int(np.sum(np.isclose(eigenvalues, 0.0))) == 2

print("degrees", degrees.tolist())
print("eigenvalues", np.round(eigenvalues, 3).tolist())


The reusable method builds an RBF or nearest-neighbor affinity, forms `L = D - A`, embeds with the smallest eigenvectors, and clusters those coordinates.

In [ ]:

X_d1 = scaled(cluster_ladder()[0][1])
labels_d1, A_d1, L_d1, vals_d1, embed_d1 = spectral_cluster(X_d1, n_clusters=3, seed=0)

print("D1 labels", labels_d1.tolist())
print("D1 first eigenvalues", np.round(vals_d1[:5], 4).tolist())
print("D1 affinity shape", A_d1.shape)


## The dataset ladder

We use the shared F2 clustering ladder exactly once in the setup cell: hand points, clean blobs, anisotropic overlap, real Iris, and real digits 0–3. The hidden labels are kept only for audit metrics such as ARI; the clustering methods never train on them.

In [ ]:

rungs = cluster_ladder()

for idx, (name, X, y_true, k) in enumerate(rungs, start=1):
    print(idx, name, "shape=", X.shape, "k=", k, "labels=", np.unique(y_true).tolist())
    print("sample", np.round(X[:3, :min(4, X.shape[1])], 3))


## Run the same method across D1–D5

The same spectral routine is run on every scaled rung. The graph is unsupervised; hidden labels are only used for ARI auditing.

In [ ]:

results = []
artifacts = []

for idx, (name, X, y_true, k) in enumerate(cluster_ladder(), start=1):
    Xs = scaled(X)
    labels, A, L, vals, embedding = spectral_cluster(Xs, n_clusters=k, seed=idx)
    score = safe_silhouette(Xs, labels)
    ari = adjusted_rand_score(y_true, labels)
    zeroish = int(np.sum(vals < 1e-6))
    results.append({"rung": idx, "name": name, "score": score, "ari": ari, "zeroish": zeroish})
    artifacts.append((Xs, labels, A, vals))

for row in results:
    print(row["rung"], row["name"], "silhouette=", round(row["score"], 3), "ARI=", round(row["ari"], 3), "near_zero_eigs=", row["zeroish"])


## Results visualization

The first panel includes the graph story through assignments on D1; all panels show cluster labels, and the curve tracks silhouette.

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(18, 3))

for ax, row, artifact in zip(axes, results, artifacts):
    Xs, labels, A, vals = artifact
    coords = plot_xy(Xs)
    ax.scatter(coords[:, 0], coords[:, 1], c=labels, cmap="tab10", s=18)
    ax.set_title(f"D{row['rung']} sil={row['score']:.2f}")
    ax.set_xticks([])
    ax.set_yticks([])

plt.show()

plt.figure(figsize=(6, 3))
plt.plot([r["rung"] for r in results], [r["score"] for r in results], marker="o")
plt.xticks([1, 2, 3, 4, 5])
plt.xlabel("ladder rung")
plt.ylabel("silhouette")
plt.title("Spectral clustering silhouette vs complexity")
plt.show()


## Pitfall on D5: affinity scale can disconnect or overconnect the graph

A nearest-neighbor graph with too few neighbors fragments digits; too many neighbors washes out local geometry. The fix is a small connectivity and silhouette sweep.

In [ ]:

name, X5, y5, k5 = cluster_ladder()[-1]
Xs5 = scaled(X5)

labels_bad, A_bad, L_bad, vals_bad, emb_bad = spectral_cluster(Xs5, n_clusters=k5, n_neighbors=1, seed=0)
bad_components = int(np.sum(vals_bad < 1e-6))
bad_score = safe_silhouette(Xs5, labels_bad)

sweep = []

for neighbors in [4, 8, 12, 20]:
    labels_fix, A_fix, L_fix, vals_fix, emb_fix = spectral_cluster(Xs5, n_clusters=k5, n_neighbors=neighbors, seed=0)
    score_fix = safe_silhouette(Xs5, labels_fix)
    components_fix = int(np.sum(vals_fix < 1e-6))
    sweep.append((score_fix, components_fix, neighbors))

best = max(sweep, key=lambda item: item[0])

print("too sparse", round(bad_score, 3), "components=", bad_components)
print("best sweep", round(best[0], 3), "components=", best[1], "neighbors=", best[2])


## Evaluate it + Practice

- Metric: track silhouette across all rungs on every rung and compare against a no-skill baseline such as random labels with the same number of clusters.
- Sanity check: D1 and D2 should be visibly easier than D5; if not, inspect scaling and assignments before trusting the curve.
- Ablation: use a graph with one neighbor or an extreme RBF gamma
- Failure signals: many near-zero eigenvalues, one giant component, or clusters that follow graph artifacts instead of data geometry
- Baseline: KMeans on raw coordinates and random labels

Practice prompts:
1. Change one hyperparameter by a small amount and explain whether the D5 score moves for a meaningful reason.

2. Add a random-label baseline to the results table and compare it with the method.

3. Pick one D5 point, print its assignment evidence, and decide whether the method is confident or ambiguous.